# Pharmagen — 1. Overview and methodology

**Pharmagen** predicts pharmacogenetic outcomes: given a patient's genotype and a
prescribed drug, it predicts the phenotype category, the direction of effect and
the type of adverse reaction.

This notebook explains the architecture and the way the project is built. The
notebooks that follow run the parts that are finished:

| Notebook | What it runs | Runtime |
|---|---|---|
| `01_overview` | Architecture, environment, repository map | CPU |
| `02_library_build` | A live drug-library build, ADMET-AI, the HGVS catalog | CPU |
| `03_graphs` | The two towers' actual graph inputs, drawn | CPU |
| `04_kg_and_evaluation` | Knowledge graph, training rows, splits, metrics | CPU |

Training is **not** demonstrated: the library was rebuilt recently, so every
model has to be retrained from scratch. `04` shows the evaluation protocol that
a trained model is measured with, and the placeholder command that launches it.

---

## Architecture: a two-tower graph neural network

```
   SMILES ──► molecular graph ──┐
   (RDKit / Modephere)          │
                                ├──► GATv2 drug tower ──┐
   conformers ──► pharmacophore ┘                       │
                   graph                                ├──► fusion ──► multi-task heads
                                                        │               (phenotype category,
   VCF / star alleles / rsIDs ──► HGVS variant graph ────┘                direction, ADR type)
   (GRCh38-validated)              GATv2 genotype tower
```

Both towers are `GATv2Tower` (`src/model/architectures/gnn.py`, PyTorch
Geometric). Their outputs are fused and fed to heads sized by the label axes.

**The drug tower** reads a molecular graph (atoms, bonds) plus a pharmacophore
graph built from one low-energy conformer — the 3D arrangement of
hydrogen-bond donors/acceptors, hydrophobes, aromatic rings and charges that
determines how a molecule actually binds.

**The genotype tower** is keyed by **HGVS**. A gene hub node carries a learned
gene-identity embedding; each HGVS change is one node whose features are read
off the notation itself, the catalog annotation and the allele dosage. Variants
are linked in genomic order (cis-adjacency). rsIDs, star alleles and PharmGKB
annotations all resolve *to* HGVS through `hgvs_catalog.pt` — nothing is keyed
on an rsID or a star label directly.

## Methodology

The project is developed spec-first. Each substantial change begins as a design
document under `docs/superpowers/specs/`, becomes a plan under
`docs/superpowers/plans/`, and only then is implemented. The specs are checked
into the repository, so the reasoning behind each architectural decision is
reviewable alongside the code:

- `2026-10-18-kg-variant-standardization-design.md` — why every KG variant is a
  canonical NCBI SPDI on GRCh38, and why indel normalization is delegated to NCBI
- `2026-10-31-evaluation-protocol-design.md` — grouped k-fold scenarios, and why
  a random split overstates performance on this data
- `2026-11-07-modephere-library-design.md` — moving the drug library into a
  separate, reusable package
- `2026-11-07-drug-sets-and-pharmacophores-design.md` — drug combinations, drug
  classes and the pharmacophore branch

Invariants that the codebase enforces rather than documents:

- **Pydantic at every boundary.** HTTP, CLI, TSV rows and TOML all parse into
  models in `src/domain/` or `src/config/`; `dict[str, Any]` is banned there.
- **Coordinates are 1-based** and **the genome build is part of every position.**
  A build mismatch is a fail-fast error, never a warning.
- **Provenance is mandatory.** Every knowledge-graph node and relationship
  carries `sources` and a `run_id`; `python -m src.kg audit` verifies it.
- **Evaluation never touches test for decisions.** Thresholds and temperature
  scaling are fitted on each fold's validation split.

Continuous integration (`.github/workflows/ci.yml`) runs `ruff check`,
`ruff format --check` and `pytest` on every push.

## Environment

Run the cell below first. On Colab it clones the repository, installs Python
3.14 with the locked dependencies (a few minutes the first time) and unpacks the
data bundle; locally it only starts the kernel over the existing checkout.

Set `BUNDLE` to wherever you put the zip produced by
`scripts/pack_colab_bundle.py`.

In [ ]:
# --- Environment -----------------------------------------------------------
# Colab runs Python 3.13; Pharmagen and Modephere need 3.14. colab_setup lets
# uv provision 3.14 + the locked dependencies and starts a kernel there. Every
# cell below marked %%pgen runs in that 3.14 kernel and keeps its state.
REPO_URL = "https://github.com/Aderfi/Pgen_dev.git"
BRANCH = "modephere-imp"

# The data bundle (scripts/pack_colab_bundle.py, ~80 MB zipped): a Google Drive path,
# or an https URL to a release asset. Ignored when running locally.
#BUNDLE = "/content/drive/MyDrive/pharmagen_colab_bundle.zip"
BUNDLE = "https://github.com/Aderfi/Pgen_dev/releases/download/demo/pharmagen_colab_bundle.zip"

import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/dev_Pgen")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
else:
    REPO, BUNDLE = Path.cwd().parent, None  # notebooks/ -> repository root

sys.path.insert(0, str(REPO / "notebooks"))
import colab_setup

kernel = colab_setup.prepare(REPO, BUNDLE)

### What was provisioned

In [ ]:
%%pgen
import sys
import torch

import polars as pl
import rdkit
import torch_geometric

from src.config import get_settings

print("Python          ", sys.version.split()[0])
print("torch           ", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("torch_geometric ", torch_geometric.__version__)
print("rdkit           ", rdkit.__version__)
print("polars          ", pl.__version__)

paths = get_settings().paths
print("\nlibrary:", paths.library)
print("data:   ", paths.data)

### Repository map

The layout below is the part of the tree these notebooks touch. `src/` is the
package; `data/` holds everything the bundle carries.

In [ ]:
%%pgen
from pathlib import Path

TREE = """
src/
├── api/          FastAPI inference service
├── cli/          python main.py {train,optuna,predict,split,evaluate}
├── config/       Pydantic Settings (settings/models/paths + TOML)
├── domain/       Pydantic domain models (Drug, Variant, Gene, Annotation)
├── data/
│   ├── cache.py      GraphCache + GraphDims (single source of graph widths)
│   ├── datasets.py   DoubleTowerDataset
│   ├── collator.py   batching, drug-set pooling indices
│   ├── genotype/     HGVS genotype tower: catalog, features, graph builder
│   └── library/      offline library build (bridge to Modephere, ADMET)
├── evaluation/   grouped k-fold protocol, metrics, bootstrap, reports
├── genomics/     star alleles, variant validation against GRCh38
├── kg/           Neo4j PharmGKB knowledge graph; the training-row source
└── model/        towers, heads, trainer, predictor, Optuna tuner

data/
├── library/      Modephere compounds + ADMET profiles + hgvs_catalog.pt
│                 pgx/ (PGx exports), kg/ (KG cache + training_rows.parquet)
├── chembl/       staged ChEMBL release (offline drug resolution)
├── dicts/        star_alleles.tsv and other static lookups
└── splits/       evaluation fold assignments, bound to the data by sha256
"""
print(TREE)

for sub in ("src", "tests"):
    files = list(Path(sub).rglob("*.py"))
    lines = sum(len(f.read_text().splitlines()) for f in files)
    print(f"{sub:6} {len(files):4} Python files, {lines:6} lines")

---

## What is complete, and what is not

**Complete and demonstrated in these notebooks**

- The drug library: Modephere builds molecular + pharmacophore graphs from a
  staged ChEMBL release, entirely offline. Notebook `02` builds one live.
- ADMET-AI profiles (41 predicted properties per compound).
- The HGVS catalog: 16,271 variants across 2,547 genes, with rsID / star-allele /
  PharmGKB resolution.
- Both towers' graph construction, end to end. Notebook `03` draws them.
- The knowledge graph: ingest with full provenance, audit, and export to
  training rows. Notebook `04`.
- The evaluation protocol: six grouped k-fold scenarios, bootstrap confidence
  intervals, calibration, baselines.
- The FastAPI inference service and the CLI.

**Known limitations** (also tracked in `CLAUDE.md`)

- Allele dosage is known for only ~30% of training rows: the source `alleles`
  field often groups several genotypes together (for example `AA + AG`).
- The genotype tower ignores whole-gene deletions. `Gene_Deletion` names no
  HGVS change, so such a genotype currently builds the same graph as the
  reference — a real gap, since a gene deletion is pharmacogenetically drastic.
- Polypharmacy is collatable but not model-ready; the switch is off.
- No trained model is shipped. The library was rebuilt, so earlier checkpoints
  no longer match the current feature widths.